# Feature Engineering: convertir el mundo en columnas sin hacer trampa

**Ciencia de Datos, Sección A** · Sesión 10 · 25 de agosto de 2026

Notebook companion de la presentación. Corran cada celda con `Shift+Enter`.

## 1. Los modelos comen números

Un modelo no ve "Quetzaltenango" ni "martes" ni "talla L": ve una matriz $X$ de números.

Las decisiones de esta sesión son **irreversibles** para el modelo: lo que no esté en $X$, no existe. Los datos ponen el techo; el modelo solo trata de alcanzarlo.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns

## 2. Variables categóricas

Codificar ciudad como `GT`=0, `QZ`=1, `ESC`=2 le dice al modelo que Escuintla está el doble de lejos que Quetzaltenango. Es falso, y el modelo se lo cree.

In [ ]:
df = pd.DataFrame({"ciudad": ["GT", "QZ", "GT"],
                   "talla": ["S", "L", "M"]})

print(pd.get_dummies(df["ciudad"], prefix="ciudad"))

# Ordinal: SOLO cuando el orden es real
orden = {"S": 0, "M": 1, "L": 2}
df["talla_num"] = df["talla"].map(orden)
print(df)

In [ ]:
# Alta cardinalidad: quedarse con el top-k y agrupar el resto
rng = np.random.default_rng(0)
muni = pd.Series(rng.choice([f"muni_{i}" for i in range(340)], size=2000))

top = muni.value_counts().head(10).index
muni_agrupado = muni.where(muni.isin(top), "OTROS")
print(muni.nunique(), "->", muni_agrupado.nunique())

### Target encoding: poderoso y peligroso

Reemplazar cada categoría por el **promedio del target** en esa categoría: una sola columna, muchísima señal.

**Por qué es peligroso:** el feature contiene el target. Si el promedio se calcula con todo el dataset, cada fila mira su propia respuesta. Se calcula **solo en train**, y conviene suavizar hacia la media global cuando la categoría tiene pocas filas.

## 3. Escalamiento

| Feature | Rango | Peso en la distancia |
|---|---|---|
| Edad (años) | 18 a 70 | despreciable |
| Ingreso (Q) | 3,000 a 90,000 | todo |

Importa para **distancias** (k-NN, k-Means, SVM), **gradientes** (regresión, redes) y **regularización**. Los **árboles no lo necesitan**: parten por umbrales, y ese orden no cambia al escalar.

In [ ]:
# Sin escalar, la distancia la decide el ingreso
a = np.array([25, 3000.0])     # edad, ingreso
b = np.array([40, 3100.0])     # +15 años, +100 Q
c = np.array([25, 12000.0])    # misma edad, +9000 Q

print(np.linalg.norm(a - b))   # la edad casi no cuenta
print(np.linalg.norm(a - c))

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split

X = rng.normal(0, 1, size=(500, 3)) * [1, 100, 0.01]
y = X[:, 0] + rng.normal(0, 0.5, 500)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, random_state=0)

sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr)
X_te_s = sc.transform(X_te)      # transform, NO fit_transform
print(X_tr_s.mean(axis=0).round(3), X_tr_s.std(axis=0).round(3))

## 4. Fechas, binning e interacciones

Un timestamp crudo es casi inútil. Lo útil es el día de la semana, el mes, si es feriado, si es quincena, cuántos días faltan para el evento.

In [ ]:
s = pd.to_datetime(["2026-08-25", "2026-09-15"])
print(s.dayofweek)     # 1, 1 (martes)
print(s.month, s.day)
print(s.dayofweek >= 5)   # ¿fin de semana?

In [ ]:
# Cíclicas: la hora 23 y la 0 están pegadas
hora = np.array([23, 0, 1])
sen = np.sin(2*np.pi*hora/24)
cos = np.cos(2*np.pi*hora/24)
print(sen.round(3))
print(cos.round(3))

# Distancia entre las 23 y las 0 en el círculo: pequeña
print(np.hypot(sen[0]-sen[1], cos[0]-cos[1]).round(3))

In [ ]:
# Binning: continuo -> categorías con sentido de negocio
edades = pd.Series([19, 33, 47, 71])
print(pd.cut(edades, bins=[0, 25, 40, 60, 120],
             labels=["joven", "adulto", "maduro", "mayor"]))

# Interacciones: el conocimiento del dominio le gana al modelo
casas = pd.DataFrame({"precio": [800000.0, 1200000.0],
                      "area": [80.0, 150.0]})
casas["precio_por_m2"] = casas["precio"] / casas["area"]
print(casas)

## 5. Leakage

**Leakage** es entrenar con información que no estará disponible al momento de predecir. El síntoma es siempre el mismo: *el modelo salió demasiado bueno*.

- **Del futuro**: predecir hospitalización usando `días_de_hospitalización`.
- **Del target**: cualquier feature construido mirando la respuesta.

Un 0.99 de accuracy no es una buena noticia: es una alarma.

In [ ]:
# MAL: el scaler ve la media y std de TODO
X_s = StandardScaler().fit_transform(X)
X_tr_mal, X_te_mal = train_test_split(X_s, random_state=0)

# BIEN: partir primero, ajustar después
X_tr_ok, X_te_ok = train_test_split(X, random_state=0)
sc = StandardScaler().fit(X_tr_ok)
X_tr_ok, X_te_ok = sc.transform(X_tr_ok), sc.transform(X_te_ok)
print(X_te_mal[:2].round(3))
print(X_te_ok[:2].round(3))

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

pipe = Pipeline([("sc", StandardScaler()),
                 ("modelo", LinearRegression())])
pipe.fit(X_tr, y_tr)          # el scaler ve solo train
print(pipe.score(X_te, y_te))

**La regla:** todo lo que *aprenda algo de los datos* (medias, desviaciones, categorías, medianas de imputación) se ajusta en train y se aplica al resto. El 17 de septiembre veremos que el Pipeline es además la única forma correcta de hacer validación cruzada.

## 6. Importancia de features

- La correlación solo ve relaciones lineales y una variable a la vez.
- Los coeficientes solo son comparables si los features están en la **misma escala**.
- Dos features muy correlacionados se reparten la importancia y ambos parecen inútiles.

Importancia no es causalidad. Guarden esa frase para el 29 de septiembre.

In [ ]:
modelo = LinearRegression().fit(X_tr_s, y_tr)
print(modelo.coef_.round(3))   # comparables porque X está escalado

## 7. Ejercicios

Completen donde dice `# ¿Qué va aquí?`.

### Ejercicio 1: encodear un dataset mixto

In [ ]:
tips = sns.load_dataset("tips")
print(tips.dtypes)

# a) one-hot para day y time
# ¿Qué va aquí?

# b) ordinal para size (el orden SÍ es real)

# c) un feature de interacción: propina por persona

# d) ¿cuántas columnas quedaron al final?

### Ejercicio 2: provocar un leakage

Midan el optimismo que produce escalar antes de partir.

In [ ]:
# a) VERSIÓN CON FUGA: escalar X completo y luego partir
# ¿Qué va aquí?  Entrenen y midan en test
score_con_fuga = ...

# b) VERSIÓN LIMPIA: partir, fit en train, transform en ambos
score_limpio = ...

# c) resten los dos scores: ese es el optimismo
# print(score_con_fuga - score_limpio)

# d) prueben también con target encoding sobre una categórica:
#    ahí la fuga es mucho mayor

### Ejercicio 3: el pipeline correcto

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# a) ColumnTransformer: OneHot a categóricas, StandardScaler a numéricas
pre = ...

# b) Pipeline: pre + un modelo
pipe = ...

# c) fit en train, score en test.
#    Comparen con el (a) del ejercicio anterior

## Lo esencial de hoy

- Categóricas: **one-hot** por defecto, ordinal solo si el orden existe, target encoding con mucho cuidado
- **Escalar** importa para distancias y gradientes; los árboles no lo necesitan
- Una fecha son muchos features; las cíclicas piden seno y coseno
- Las **interacciones** del dominio le ganan a los modelos
- **Leakage**: si el resultado es demasiado bueno, búsquenlo
- La regla que resume todo: **fit solo en train**, y Pipeline para no olvidarlo

**Próxima clase (jueves 27): k-NN y Perceptrón.** Hoy se entrega la HDT 3.